Collect the patches with high attribution for your kernel.  

- Collect all attribution map outputs of a kernel
- Go through each POIs scatter plots and find the threshold
  - alternatively, its much easier to just exclude the first row and col (the paddings)
  - Hopefully the model learns enough
  - They have bad distributions
  - Then use otsu for finding thresholds, separately for positive and negative values
- go through all activations
  - if the attribution for a patch > thresh, save it
  - Saved as a bunch of pt files (one per batch)

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    otsu_threshold,
    zeros_with_1_at,
)
from pt_to_api.utils import *
from pt_to_api.capture import get_model_internals
from pt_to_api.mnist import (
    SimpleMNIST,
    get_contribs_for_inp_vectorized,
    get_mnist_dataloader,
)
from tqdm import tqdm
import gc

# Helpers

In [ ]:
import seaborn as sns



def scatter_plot_1d(numbers, suff="", vlines=None):
    plt.figure(figsize=(20, 3))
    sns.stripplot(x=numbers, color="blue", alpha=0.5, jitter=True)
    
    
    if vlines:
        colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
        for i, (label, x) in enumerate(vlines.items()):
            plt.axvline(x=x, linestyle="--", label=label, color=colors[i % len(colors)])
        plt.legend()
    
    plt.title("1D Clustering Visualization" + suff)
    plt.xlabel("Value")
    plt.grid(axis="x", linestyle="--", alpha=0.6)
    plt.show()

# Load model

In [ ]:
MODEL_PATH = Path("../../../pt-to-api/data/model.pt")
INPUT_PATH = Path("../../../pt-to-api/data/first-input-tens.pt")

DRIVE_PATH = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist")

MAIN_OUT_DIR = (DRIVE_PATH / "collect-patches" / "data")
MAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)
device = "mps"

inp = torch.load(INPUT_PATH, weights_only=False)
model = SimpleMNIST()
model.load_state_dict(torch.load(MODEL_PATH))
model = model.to(device)

# Collection start


All configuration below, change it to collect the patches of the kernel you want.   

In [ ]:
import torch.nn.functional as F
from torch import nn

def get_indices_of_patches_to_extract(contribs, layer_name, channel, pos_threshes, neg_threshes):
    # print("slice", .shape, "thres", pos_threshes.shape, "neg", neg_threshes.shape)
    contrib_slice = contribs[layer_name][:, channel]
    pos_inds = torch.argwhere(contrib_slice >= pos_threshes)
    neg_inds = torch.argwhere(contrib_slice <= neg_threshes)
    return torch.cat([pos_inds, neg_inds])


def patches_of_single_batch_with_indices(input_act_of_batch, layer, indices):
    op_shape = get_output_shape(
        input_act_of_batch.shape, layer.kernel_size, layer.stride, layer.padding, layer.dilation
    )
    op_r, op_c = op_shape[-2], op_shape[-1]
    b = input_act_of_batch.shape[0]

    patches = F.unfold(
        input_act_of_batch, layer.kernel_size, layer.dilation, layer.padding, layer.stride
    ).reshape(b, -1, op_r, op_c)

    return torch.stack([
        patches[ind[0], :, ind[1], ind[2]]
        for ind in indices
    ])


def get_output_shape(input_shape, ksize, stride, padding, dilation):
    B, C, H, W = input_shape
    Ho = (H + 2*padding[0] - dilation[0]*(ksize[0]-1) - 1) // stride[0] + 1
    Wo = (W + 2*padding[1] - dilation[1]*(ksize[1]-1) - 1) // stride[1] + 1
    return (B, C, Ho, Wo)


In [ ]:
def get_sampled_patches(patches_ds, num_samples=-1):
    patches_ds = Path(patches_ds)
    patches = []
    for p in patches_ds.glob("*.pt"):
        patches.append(torch.load(p, weights_only=False, map_location="cpu").numpy())
    patches = np.concat(patches)
    samples_idxs = torch.randperm(patches.shape[0]).numpy()
    total = len(samples_idxs)
    to_extract = num_samples if num_samples != -1 else total

    samples = patches[samples_idxs[:to_extract]]
    return samples


def save_weight_and_patches(model, layer_name, channel, src_dir, out_dir, n_samples=-1):
    layer = model.get_submodule(f"{layer_name}")
    weight = layer.weight[channel].clone().detach().cpu()
    weight = weight.reshape(-1).numpy()
    to_save_patches = get_sampled_patches(src_dir, n_samples)

    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)

    torch.save(to_save_patches, out_dir / "samples.pt")
    torch.save(weight, out_dir / f"weight.pt")

In [ ]:
def get_contribs_for_batch(batch, targets, device):
    targs = torch.concat([zeros_with_1_at(10, targ) for targ in targets]).to(device)
    contribs, acts, params = get_contribs_for_inp_vectorized(
        batch, model, targs, "layers.5", device
    )
    return contribs, acts, params


def get_stds_and_bad_combs(collected_contribs):
    all_stds = []
    bad_combs = []

    for r in range(collected_contribs.shape[1]):
        for c in range(collected_contribs.shape[2]):
            sm = collected_contribs[:, r, c]
            all_stds.append(sm.std().item())
            if sm.std() < 2e-4 and sm.mean().abs() < 1e-4:
                bad_combs.append((r, c))
    return all_stds, bad_combs


def get_pos_and_neg_threshes(collected_contribs, bad_combs):
    INF_POS_CONTRIB, INF_NEG_CONTRIB = 2, -2

    pos_threshes = np.zeros_like(collected_contribs[0])
    neg_threshes = np.zeros_like(collected_contribs[0])


    pos_threshes.fill(INF_POS_CONTRIB)
    neg_threshes.fill(INF_NEG_CONTRIB)


    for r in range(collected_contribs.shape[1]):
        for c in range(collected_contribs.shape[2]):
            if (r, c) in bad_combs:
                print("not relevant", r, c)
            else:
                vals = collected_contribs[:, r, c].numpy()
                pvals = [v for v in vals if v > 0]
                nvals = [v for v in vals if v < 0]
                pthres, nthres = otsu_threshold(pvals), otsu_threshold(nvals)
                pos_threshes[r, c] = pthres
                neg_threshes[r, c] = nthres
    pos_threshes, neg_threshes = torch.tensor(pos_threshes), torch.tensor(neg_threshes)
    return pos_threshes, neg_threshes

In [ ]:
def get_collected_contribs(dl, layer_key, channel, device):
    collected_contribs = []
    for batch, targets in tqdm(dl.train):
        contribs, _, _ = get_contribs_for_batch(batch, targets, device)
        collected_contribs.append(contribs[layer_key][:, channel])
    collected_contribs = torch.concat(collected_contribs)
    print(f"collected contribs for layer={layer_key} channel={channel}")
    return collected_contribs


def single_cycle(model, layer_key, channel, input_act_key, patches_out_dir, device):
    layer = model.get_submodule(layer_key)
    print("############ start contrib collection")
    dl = get_mnist_dataloader(0.2, bs=256)
    collected_contribs = get_collected_contribs(dl, layer_key, channel, device)

    print("########## find combs")
    all_stds, bad_combs = get_stds_and_bad_combs(collected_contribs)

    print("######## find and save thresholds")
    pos_threshes, neg_threshes = get_pos_and_neg_threshes(collected_contribs, bad_combs)
    torch.save(pos_threshes, patches_out_dir / "pos_threshes.pt")
    torch.save(neg_threshes, patches_out_dir / "neg_threshes.pt")

    import gc
    print("gc: released", gc.collect())


    print("########## start collecting patches")
    main_dl = get_mnist_dataloader(1, bs=1024)
    out_dir = patches_out_dir / "patches"
    out_dir.mkdir(exist_ok=True, parents=True)

    for i, (batch, targets) in enumerate(tqdm(main_dl.train)):
        contribs, acts, params = get_contribs_for_batch(batch, targets, device)
        indices = get_indices_of_patches_to_extract(contribs, layer_key, channel, pos_threshes, neg_threshes)
        patches_of_batch = patches_of_single_batch_with_indices(acts[input_act_key], layer, indices)
        torch.save(patches_of_batch, out_dir / f"{i}.pt")

    print("############# save patches")
    save_weight_and_patches(model, layer_key, channel, out_dir, patches_out_dir, subset_size)

# Calculate contribs

In [ ]:
layer_key = "layers.0"
input_act_key = "x"
layer = model.get_submodule(layer_key)
subset_size = 10_000

In [ ]:
for channel in range(0, 8):
    patches_out_dir = MAIN_OUT_DIR / layer_key / str(channel)
    patches_out_dir.mkdir(parents=True, exist_ok=True)
    single_cycle(model, layer_key, channel, input_act_key, patches_out_dir, device)
    print("gc: released", gc.collect(), "bytes")

## Minor testing

In [ ]:
w = torch.load(patches_out_dir / "weight.pt", weights_only=False)
print("all close", torch.allclose(layer.weight[channel].cpu().reshape(-1), torch.tensor(w)))
samples = torch.load(patches_out_dir / "samples.pt", weights_only=False)
S([s.reshape(8,9) for s in samples[:5]], (20,4), 5)
plt.show()

In [ ]:
# how do i test that the patches are good?

In [ ]:
pos_threshes = torch.load(patches_out_dir / "pos_threshes.pt")
neg_threshes = torch.load(patches_out_dir / "neg_threshes.pt")
# torch.save(pos_threshes, )
# torch.save(neg_threshes, patches_out_dir / "neg_threshes.pt")

In [ ]:
print("########## start collecting patches")
main_dl = get_mnist_dataloader(1, bs=1024)
# out_dir = patches_out_dir / "patches"
# out_dir.mkdir(exist_ok=True, parents=True)

batch, targets = next(iter(main_dl.train))
# for i, (batch, targets) in enumerate(tqdm(main_dl.train)):
contribs, acts, params = get_contribs_for_batch(batch, targets, device)
indices = get_indices_of_patches_to_extract(contribs, layer_key, channel, pos_threshes, neg_threshes)
patches_of_batch = patches_of_single_batch_with_indices(acts[input_act_key], layer, indices)

# torch.save(patches_of_batch, out_dir / f"{i}.pt")

In [ ]:
acts[input_act_key].shape

In [ ]:
indices

In [ ]:
pos_threshes

In [ ]:
S(tsl(acts[input_act_key][1]), (20,4), ncols=8)

In [ ]:
# image 1, row 2, col 3
indices[0]

S([acts[input_act_key][1][:, 2:5, 3:6].reshape(8,9)])

In [ ]:
patches_of_batch.shape, indices.shape

In [ ]:
indices